# 🧠 FlyWire 2026 Challenge — Full Pipeline
**Memory-efficient implementation for free Colab (15GB RAM)**

Key design choices:
- Uses `scipy.sparse` NOT NetworkX → ~200MB total for all 5 graphs
- Loads graphs one at a time when possible
- Uses set operations for fast intersection computation

**Before running:**
1. Upload all 5 CSV files to Google Drive under `My Drive/flywire_challenge/`
2. Set runtime to GPU: Runtime → Change runtime type → T4 GPU
3. Run cells top to bottom

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: Mount Drive & Install Libraries
# ═══════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

# Install only what we need (all very small, fast install)
!pip install -q scipy numpy pandas

import os, time
import numpy as np
import pandas as pd
import scipy.sparse as sp

DATA = '/content/drive/MyDrive/flywire_challenge/'
print('Files available:', os.listdir(DATA))

# Check RAM available
import psutil
ram = psutil.virtual_memory()
print(f'RAM: {ram.total/1e9:.1f} GB total, {ram.available/1e9:.1f} GB free')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: Load All 5 Graphs as Scipy Sparse Matrices
# Total memory: ~200MB (not GB!) — fits easily in 15GB
# ═══════════════════════════════════════════════════════════

DATASETS = {
    'FAFB': 'fafb_783_edge_list.csv',
    'BANC': 'banc_626_edge_list.csv',
    'MANC': 'manc_1.2.1_edge_list.csv',
    'MAOL': 'maol_1.1_edge_list.csv',
    'MCNS': 'mcns_0.9_edge_list.csv',
}

def load_graph(name, fname):
    """Load edge list → scipy sparse adjacency matrix + node mappings."""
    t = time.time()
    df = pd.read_csv(DATA + fname)
    df.columns = ['src', 'tgt']
    df = df[df['src'] != df['tgt']].drop_duplicates()  # remove self-loops

    # Map node IDs to compact integers 0..N-1
    all_nodes = sorted(set(df['src']) | set(df['tgt']))
    n2i = {n: i for i, n in enumerate(all_nodes)}  # node → index
    i2n = np.array(all_nodes)                       # index → node
    N = len(all_nodes)

    rows = df['src'].map(n2i).values
    cols = df['tgt'].map(n2i).values
    data = np.ones(len(rows), dtype=np.bool_)

    mat = sp.csr_matrix((data, (rows, cols)), shape=(N, N), dtype=np.bool_)
    mem_mb = (mat.data.nbytes + mat.indices.nbytes + mat.indptr.nbytes) / 1e6

    print(f'{name}: {N:>7,} nodes | {mat.nnz:>8,} edges | {mem_mb:.1f} MB sparse | {time.time()-t:.1f}s')
    return {'mat': mat, 'n2i': n2i, 'i2n': i2n, 'df': df, 'name': name}

graphs = {}
for name, fname in DATASETS.items():
    graphs[name] = load_graph(name, fname)

import psutil
print(f'\nRAM used: {psutil.Process().memory_info().rss / 1e9:.2f} GB')
print('✅ All 5 graphs loaded!')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3: STRATEGY A — MAOL + MCNS Direct Match (Free Shortcut!)
#
# DISCOVERY: 99.9% of MAOL node IDs appear in MCNS by same integer ID.
# These are the SAME physical neurons (optic lobe), just different
# synapse thresholds (MAOL: 1+, MCNS: 5+).
#
# Algorithm:
#   Find all neuron pairs (u, v) where BOTH are in MAOL and MCNS,
#   and the edge (u→v) status is IDENTICAL in both graphs.
#   The largest such set = our MAOL∩MCNS common induced subgraph.
# ═══════════════════════════════════════════════════════════

print('=== STRATEGY A: MAOL + MCNS Direct Match ===')
t = time.time()

maol_nodes = set(graphs['MAOL']['n2i'].keys())
mcns_nodes = set(graphs['MCNS']['n2i'].keys())
shared = maol_nodes & mcns_nodes
print(f'Shared node IDs: {len(shared):,} ({100*len(shared)/len(maol_nodes):.1f}% of MAOL)')

# Build edge sets for shared nodes
maol_df = graphs['MAOL']['df']
mcns_df = graphs['MCNS']['df']

# Filter edges to only those between shared nodes
maol_int = maol_df[maol_df['src'].isin(shared) & maol_df['tgt'].isin(shared)]
mcns_int = mcns_df[mcns_df['src'].isin(shared) & mcns_df['tgt'].isin(shared)]

maol_set = set(zip(maol_int['src'], maol_int['tgt']))
mcns_set = set(zip(mcns_int['src'], mcns_int['tgt']))

agree = maol_set & mcns_set        # edge in BOTH  → OK
maol_only = maol_set - mcns_set    # edge in MAOL only → conflict (weak synapse)
mcns_only = mcns_set - maol_set    # edge in MCNS only → conflict (seg difference)

print(f'Edges in both (5+ synapse):  {len(agree):,}')
print(f'MAOL-only (1-4 syn, weak):   {len(maol_only):,}  ← causes conflict')
print(f'MCNS-only (seg difference):  {len(mcns_only):,}  ← causes conflict')
print(f'% of MCNS edges also in MAOL: {100*len(agree)/max(1,len(mcns_set)):.1f}%')

# Neurons involved in any conflict — can't all be in the common subgraph together
conflict_nodes = set()
for s, t in maol_only: conflict_nodes.add(s); conflict_nodes.add(t)
for s, t in mcns_only: conflict_nodes.add(s); conflict_nodes.add(t)
safe_nodes = shared - conflict_nodes

# Edges within safe nodes
safe_edges = {(s,t) for s,t in agree if s in safe_nodes and t in safe_nodes}

print(f'\n🎯 MAOL+MCNS common induced subgraph:')
print(f'   Neurons: {len(safe_nodes):,}')
print(f'   Edges:   {len(safe_edges):,}')
print(f'   Time: {time.time()-t:.1f}s')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4: STRATEGY B — FAFB + BANC + MCNS (Best Overall)
#
# BANC has pre-computed matches to FAFB neurons (fafb_783_match_id).
# This is the most biologically meaningful approach.
#
# To get BANC metadata:
#   Option 1: Download from Codex → codex.flywire.ai/api/download
#   Option 2: Use CAVEclient (see below)
#   Option 3: Harvard Dataverse (requires finding exact DOI)
# ═══════════════════════════════════════════════════════════

# Try CAVEclient to fetch BANC metadata directly
!pip install -q caveclient
from caveclient import CAVEclient

# Initialize the BANC client (public, no auth needed for some tables)
try:
    client = CAVEclient('brain_and_nerve_cord')
    print('Available annotation tables:')
    print(client.annotation.get_tables())
except Exception as e:
    print(f'CAVEclient error: {e}')
    print('Try: client = CAVEclient("minnie65_phase3_v1") for FAFB')
    print('Or download BANC metadata manually from:')
    print('https://codex.flywire.ai/api/download  (sign in first)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5: Degree-Based Fingerprinting (No Metadata Needed)
#
# Even without cell type labels, we can create structural
# fingerprints for each neuron based on its connection pattern.
#
# A neuron's fingerprint = sorted list of its neighbours' degrees.
# Two neurons with the same fingerprint are candidates for matching.
# This works especially well for neurons with unique degree patterns.
# ═══════════════════════════════════════════════════════════

def compute_fingerprints(mat, i2n, depth=1):
    """
    Compute structural fingerprint for each neuron.
    depth=1: use just (in_degree, out_degree)
    depth=2: also include degree-of-neighbours (richer, slower)
    """
    N = mat.shape[0]
    in_deg  = np.array(mat.sum(axis=0)).flatten()   # how many inputs
    out_deg = np.array(mat.sum(axis=1)).flatten()   # how many outputs

    if depth == 1:
        # Simple: (in_degree, out_degree)
        fps = list(zip(in_deg.astype(int), out_deg.astype(int)))
    else:
        # Richer: sorted tuple of neighbour in-degrees
        fps = []
        for i in range(N):
            # Get indices of outgoing neighbours
            nbrs = mat[i].indices
            nbr_in_degs = tuple(sorted(in_deg[nbrs].astype(int)))
            fps.append((int(in_deg[i]), int(out_deg[i]), nbr_in_degs))

    return pd.DataFrame({'node_id': i2n, 'fingerprint': fps})

print('Computing fingerprints for FAFB...')
t = time.time()
fafb_fps = compute_fingerprints(graphs['FAFB']['mat'], graphs['FAFB']['i2n'])
print(f'Done in {time.time()-t:.1f}s')
print(f'Unique fingerprints: {fafb_fps["fingerprint"].nunique():,} / {len(fafb_fps):,} neurons')
print('\nTop 10 most common degree patterns (in_deg, out_deg):')
print(fafb_fps['fingerprint'].value_counts().head(10))
print('\nUnique fingerprints = candidate for 1-to-1 matching!')
unique_fps = fafb_fps['fingerprint'].value_counts()
print(f'Neurons with UNIQUE fingerprint: {(unique_fps == 1).sum():,}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6: Cross-Dataset Fingerprint Matching
#
# Match neurons across datasets by their structural fingerprint.
# Neurons with identical fingerprints are isomorphism candidates.
# Neurons with UNIQUE fingerprints across both datasets are
# automatically matched (only one possibility exists!)
# ═══════════════════════════════════════════════════════════

def match_by_fingerprint(fps_a, fps_b):
    """
    Find candidate node pairs (u from A, v from B) where fingerprints match.
    Returns:
      - exact_matches: fingerprint appears exactly once in both → auto-matched!
      - candidates: fingerprint appears in both but multiple times → need further check
    """
    # Count occurrences of each fingerprint in each dataset
    count_a = fps_a['fingerprint'].value_counts()
    count_b = fps_b['fingerprint'].value_counts()

    # Fingerprints that appear in both
    common_fps = set(count_a.index) & set(count_b.index)
    print(f'Fingerprints in A: {len(count_a):,}')
    print(f'Fingerprints in B: {len(count_b):,}')
    print(f'Shared fingerprints: {len(common_fps):,}')

    # Exact matches: fingerprint unique in BOTH datasets
    exact_fps = {fp for fp in common_fps if count_a[fp] == 1 and count_b[fp] == 1}
    print(f'Unique (auto-matched) fingerprints: {len(exact_fps):,}')

    exact_a = fps_a[fps_a['fingerprint'].isin(exact_fps)].set_index('fingerprint')
    exact_b = fps_b[fps_b['fingerprint'].isin(exact_fps)].set_index('fingerprint')
    exact_matches = exact_a.join(exact_b, lsuffix='_A', rsuffix='_B')
    exact_matches = exact_matches.reset_index()
    exact_matches.columns = ['fingerprint', 'node_A', 'node_B']

    # Candidate matches: fingerprint in both but not unique
    ambig_fps = common_fps - exact_fps
    total_candidates = sum(count_a[fp] * count_b[fp] for fp in ambig_fps)
    print(f'Ambiguous fingerprints: {len(ambig_fps):,} (generating {total_candidates:,} candidate pairs)')

    return exact_matches

print('=== Matching FAFB vs MCNS by fingerprint ===')
print('Computing MCNS fingerprints...')
mcns_fps = compute_fingerprints(graphs['MCNS']['mat'], graphs['MCNS']['i2n'])

print('\nMatching...')
fafb_mcns_matches = match_by_fingerprint(fafb_fps, mcns_fps)
print(f'\nAuto-matched FAFB↔MCNS pairs: {len(fafb_mcns_matches):,}')
print(fafb_mcns_matches.head())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7: Verify Common Induced Subgraph
#
# Given a set of matched pairs (u_A, u_B) between datasets A and B,
# verify that the induced subgraph is actually isomorphic.
#
# For each pair of matched pairs (u_A↔u_B) and (v_A↔v_B):
#   edge (u_A→v_A) exists in A  ←→  edge (u_B→v_B) exists in B
# ═══════════════════════════════════════════════════════════

def verify_isomorphism(matches_df, g_a, g_b):
    """
    Verify that the matched nodes form isomorphic induced subgraphs.
    Returns: (is_valid, num_valid_pairs, violations)
    """
    nodes_a = matches_df['node_A'].tolist()
    nodes_b = matches_df['node_B'].tolist()

    # Get indices in each graph
    idx_a = [g_a['n2i'][n] for n in nodes_a if n in g_a['n2i']]
    idx_b = [g_b['n2i'][n] for n in nodes_b if n in g_b['n2i']]

    # Extract induced submatrices (sub-adjacency matrices)
    sub_a = g_a['mat'][np.ix_(idx_a, idx_a)].toarray()
    sub_b = g_b['mat'][np.ix_(idx_b, idx_b)].toarray()

    # Check if identical
    match = (sub_a == sub_b)
    violations = (~match).sum()
    is_valid = violations == 0

    print(f'Subgraph size: {len(idx_a)} × {len(idx_a)}')
    print(f'Edge agreements: {match.sum():,} / {match.size:,}')
    print(f'Violations: {violations}')
    print(f'Isomorphic: {"✅ YES" if is_valid else "❌ NO — need to prune conflicts"}')

    return is_valid, violations, sub_a, sub_b

# Verify on first 500 auto-matched pairs as a test
test_matches = fafb_mcns_matches.head(500)
print(f'Verifying {len(test_matches)} auto-matched FAFB↔MCNS pairs...')
valid, violations, sub_a, sub_b = verify_isomorphism(test_matches, graphs['FAFB'], graphs['MCNS'])

if not valid:
    print('\nNote: fingerprint matching is a heuristic — some conflicts expected')
    print('Will apply conflict-pruning in next cell')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8: Iterative Conflict Pruning
#
# If the matched set has violations, iteratively remove the
# most conflicted node until the set is fully isomorphic.
# This is a greedy approach to maximizing the valid set.
# ═══════════════════════════════════════════════════════════

def prune_to_isomorphism(matches_df, g_a, g_b, max_iter=1000):
    """
    Iteratively remove the node involved in the most conflicts
    until the induced subgraph is perfectly isomorphic.
    Returns the largest valid matched set.
    """
    current = matches_df.copy().reset_index(drop=True)
    iteration = 0

    while iteration < max_iter:
        nodes_a = current['node_A'].tolist()
        nodes_b = current['node_B'].tolist()

        # Only keep nodes that exist in both graphs
        valid_mask = [(n in g_a['n2i'] and n in g_b['n2i'])
                      for n, m in zip(nodes_a, nodes_b)]
        # Actually check node_A in g_a and node_B in g_b
        valid_mask = [(na in g_a['n2i'] and nb in g_b['n2i'])
                      for na, nb in zip(nodes_a, nodes_b)]
        current = current[valid_mask].reset_index(drop=True)
        nodes_a = current['node_A'].tolist()
        nodes_b = current['node_B'].tolist()

        idx_a = [g_a['n2i'][n] for n in nodes_a]
        idx_b = [g_b['n2i'][n] for n in nodes_b]

        sub_a = g_a['mat'][np.ix_(idx_a, idx_a)].toarray()
        sub_b = g_b['mat'][np.ix_(idx_b, idx_b)].toarray()

        diff = (sub_a != sub_b)
        if diff.sum() == 0:
            print(f'✅ Converged after {iteration} removals!')
            print(f'Final common induced subgraph: {len(current)} neurons')
            return current

        # Find the node involved in the most violations
        row_violations = diff.sum(axis=1) + diff.sum(axis=0)
        worst_idx = np.argmax(row_violations)
        current = current.drop(index=worst_idx).reset_index(drop=True)
        iteration += 1

        if iteration % 50 == 0:
            print(f'  iter {iteration}: {len(current)} nodes remaining, {diff.sum()} violations')

    print(f'Did not converge in {max_iter} iterations')
    return current


print('Pruning to isomorphism...')
t = time.time()
valid_matches = prune_to_isomorphism(fafb_mcns_matches, graphs['FAFB'], graphs['MCNS'])
print(f'Time: {time.time()-t:.1f}s')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9: Extend to 3rd Dataset
#
# We have valid FAFB↔MCNS matches.
# Now find the same neurons in BANC (or MAOL).
# BANC metadata gives us fafb_783_match_id → direct BANC↔FAFB mapping.
# ═══════════════════════════════════════════════════════════

# Load BANC metadata if available
BANC_META = DATA + 'banc_metadata.csv'
if os.path.exists(BANC_META):
    banc_meta = pd.read_csv(BANC_META)
    print('BANC metadata columns:', list(banc_meta.columns))

    # Find cross-dataset match column
    fafb_col = [c for c in banc_meta.columns if 'fafb' in c.lower()]
    print('FAFB match columns:', fafb_col)

    if fafb_col:
        # Build BANC↔FAFB mapping
        banc_id_col = 'root_id' if 'root_id' in banc_meta.columns else banc_meta.columns[0]
        fafb_map = banc_meta[[banc_id_col, fafb_col[0]]].dropna()
        fafb_to_banc = dict(zip(fafb_map[fafb_col[0]], fafb_map[banc_id_col]))
        print(f'FAFB→BANC mappings: {len(fafb_to_banc):,}')

        # Find which of our matched FAFB neurons have a BANC counterpart
        final_triplets = []
        for _, row in valid_matches.iterrows():
            fafb_id = row['node_A']
            mcns_id = row['node_B']
            if fafb_id in fafb_to_banc:
                banc_id = fafb_to_banc[fafb_id]
                if banc_id in graphs['BANC']['n2i']:
                    final_triplets.append({'FAFB': fafb_id, 'BANC': banc_id, 'MCNS': mcns_id})

        triplets_df = pd.DataFrame(final_triplets)
        print(f'\n🎯 Triplets (FAFB + BANC + MCNS): {len(triplets_df):,} neurons!')
        print(triplets_df.head())
else:
    print('BANC metadata not found — skipping BANC extension')
    print('Upload banc_metadata.csv to your Drive folder')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10: Final Verification + Generate Submission CSV
# ═══════════════════════════════════════════════════════════

def final_verify(triplets_df, graphs, d1='FAFB', d2='BANC', d3='MCNS'):
    """Full verification that the 3-way induced subgraph is isomorphic."""
    nodes = {d1: triplets_df[d1].tolist(),
             d2: triplets_df[d2].tolist(),
             d3: triplets_df[d3].tolist()}

    mats = {}
    for d in [d1, d2, d3]:
        g = graphs[d]
        idx = [g['n2i'][n] for n in nodes[d] if n in g['n2i']]
        mats[d] = g['mat'][np.ix_(idx, idx)].toarray()

    ok_12 = (mats[d1] == mats[d2]).all()
    ok_13 = (mats[d1] == mats[d3]).all()
    ok_23 = (mats[d2] == mats[d3]).all()

    print(f'{d1} == {d2}: {"✅" if ok_12 else "❌"}')
    print(f'{d1} == {d3}: {"✅" if ok_13 else "❌"}')
    print(f'{d2} == {d3}: {"✅" if ok_23 else "❌"}')

    return ok_12 and ok_13 and ok_23

# Save submission CSV
def save_submission(triplets_df, d1='FAFB', d2='BANC', d3='MCNS'):
    out = triplets_df[[d1, d2, d3]].copy()
    out.columns = [d1, d2, d3]
    out.to_csv(DATA + 'submission.csv', index=False)
    print(f'Saved submission.csv with {len(out):,} neurons')
    print('Preview:')
    print(out.head())

# Uncomment when ready:
# verified = final_verify(triplets_df, graphs)
# if verified:
#     save_submission(triplets_df)
print('Run final_verify() and save_submission() once triplets are ready!')